# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and step-by-step approach for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL from the SenScience FAIR² repository.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing top level dataset info
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect the record sets defined in the Croissant package, and for each, list the available fields (with their `@id`).

In [ ]:
# List all record sets and their fields by @id
def print_record_sets(dataset):
    if not hasattr(dataset.metadata, "recordSet") or not dataset.metadata.recordSet:
        print("No record sets found in metadata.")
        return []
    
    record_sets = dataset.metadata.recordSet
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    rs_ids = []
    for rs in record_sets:
        rs_id = getattr(rs, '@id', rs.get('@id', None))
        rs_ids.append(rs_id)
        print(f"RecordSet '@id': {rs_id}")
        print(f"  Name: {getattr(rs, 'name', None)}")
        print(f"  Description: {getattr(rs, 'description', None)}")
        # List fields
        fields = getattr(rs, 'field', None)
        if fields is not None:
            fields_list = fields if isinstance(fields, list) else [fields]
            print("  Fields & Columns:")
            for field in fields_list:
                field_id = getattr(field, '@id', field.get('@id', None))
                print(f"    - Field @id: {field_id}")
        print()
    return rs_ids

# Get all record sets in this dataset
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_set_ids = print_record_sets(dataset)
else:
    print('No record sets declared in the metadata. Inspecting underlying DataFiles as fallback...')
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        # For many experimental data packages, there may be implicit or nonstandard recordSets
        for dist in dataset.metadata.distribution:
            dist_id = getattr(dist, '@id', dist.get('@id', None))
            print(f"Distribution @id: {dist_id}")
    else:
        print('No distribution or data files found.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

If there were no explicit record sets, we attempt to infer them directly from the Croissant package's data files. In this dataset, the records appear in a main tabular data file corresponding to the `distribution` section.

In [ ]:
# Gather available record sets for loading
# If no recordSet section, examine distribution for tabular files

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

record_set_ids = []
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    for rs in record_sets:
        rs_id = getattr(rs, '@id', rs.get('@id', None))
        record_set_ids.append(rs_id)
else:
    # Try to extract DataFiles by distribution as @id
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        for dist in dataset.metadata.distribution:
            dist_id = getattr(dist, '@id', dist.get('@id', None))
            record_set_ids.append(dist_id)

if not record_set_ids:
    raise ValueError("No record sets or datafile @id found. Please check the Croissant schema.")

# List of discovered record sets or main data files by @id
print("Record sets/data files found (referenced by @id):", record_set_ids)

# Attempt loading all record sets/datafiles as Croissant records
dataframes = {}
for rs_id in record_set_ids:
    print(f"\nExtracting records for record set or data file: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            print(f"  Columns in {rs_id}:\n    {df.columns.tolist()}")
            print(f"  Preview (first 5 rows):")
            display(df.head())
        else:
            print('  No records fetched (empty).')
        dataframes[rs_id] = df
    except Exception as e:
        print(f"  Failed to load records: {str(e)}")

# Choose the first non-empty record set for downstream analysis
for main_rs_id, main_df in dataframes.items():
    if not main_df.empty:
        break
else:
    raise ValueError("No non-empty record sets found for analysis.")

print(f"\nMain DataFrame columns: {main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply several typical data processing steps:
- Filter records based on a specific quantitative field (for example, filter proteins with coverage percentage above a threshold)
- Normalize numeric columns
- Group data by a categorical variable (if present)

All column references use their **exact @id** as in the Croissant metadata.

First, we inspect columns for likely numeric fields (e.g., 'coverage', 'peptide_count', 'MW').

In [ ]:
# List numeric candidate columns (@id in Croissant schema), pick one for thresholding & normalization
possible_numeric_fields = [col for col in main_df.columns if main_df[col].dtype.kind in 'fi' or pd.api.types.is_numeric_dtype(main_df[col])]
print("Possible numeric fields:", possible_numeric_fields)

# Choose a common field for demonstration; user can change this based on the dataset
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0] # e.g., 'coverage_percent', or similar
else:
    raise ValueError('No numeric fields found for EDA step.')

print(f"Using {numeric_field} for numeric analysis.")

# Choose threshold (e.g., 10, default)
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Suggest a group field (try 'description', 'sample', 'accession', etc.)
group_candidates = [col for col in main_df.columns if main_df[col].dtype == 'O']
print('Categorical / Groupable fields:', group_candidates)

group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing means):")
    display(grouped_df.head())
else:
    print('No suitable categorical field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields.

Let's plot the distribution of the selected numeric field, and a boxplot grouped by the selected categorical field (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field} (Filtered > {threshold})')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

if group_field:
    plt.figure(figsize=(10,4))
    # Limit to top 10 groups for visibility
    top_groups = filtered_df[group_field].value_counts().head(10).index
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(top_groups)])
    plt.title(f'{numeric_field} by {group_field} (top 10 groups)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a [Croissant FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) with the `mlcroissant` Python library. We reviewed metadata, extracted data records by their `@id`, performed filtering and normalization on numeric fields, grouped data using categorical annotations, and visualized data distributions.

This workflow is general for datasets with rich metadata in the Croissant schema, ensuring transparent, reproducible, and structured data processing pipelines. For more advanced analyses, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant).